## Transferring Data from Device manufacturer sites to Influxdb
For flexibility and easy of data manipulation, we transfer the data to influx db.
This will also enable us to have the data downloaded and stored in one place - giving us ownership incase our subscription with the device manufacturer expires.

### Prepare tools for the project
First install the packages and modules necessary to implement the steps of the project.

The documentation for the influxdb_client_3 library can be found [here](https://docs.influxdata.com/influxdb3/cloud-dedicated/reference/client-libraries/v3/python/)

In [1]:
#Install operational packages to help in 
#organizing and querrying the data 
import pandas as pd
import numpy as np
import requests 
import json

#Modules to help interact with influxdb
import os, time
from influxdb_client_3 import InfluxDBClient3, Point


Create a magic function to help in skipping cells so that they run only once. Some cells especially those that involve uplaoding data to the database, can be skipped so that the data is not duplicated when the cells are are run multiple times

In [2]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

### Set up influxdb
Prepare the database to receive data from the device manufacturer urls 

In [3]:
#add this to load and upadte the token file
%reload_ext dotenv
%dotenv

#Define some of the arguments required 
#to write the data to the database

token = os.environ.get("INFLUXDB_TOKEN")
org         =   "Air quality - Harrisburg and Philadelphia"
host        =   "https://us-east-1-1.aws.cloud2.influxdata.com"
database    =   "pilot-harrisburg-home"

client = InfluxDBClient3(host=host, token=token, org=org, database=database)

### Upload the data from Devices to influxdb
Make API calls to the device urls and send the data to the influxdb database. Since different manufacturers have different procedures for API calls, we do upload the data in different batches based on the manufacturer

1. QUANTAQ

The guidelines to make API calls to quantaq can be found [here](https://docs.quant-aq.com/software-apis-and-libraries/quantaq-cloud-api)

In [17]:
#Load the api key and device serial numbers from the envrionment variable 
%reload_ext dotenv
%dotenv
QUANTAQ_APIKEY = os.environ.get("QUANTAQ_APIKEY")
QUANT001_SN = os.environ.get("QUANT001_SN")
QUANT002_SN = os.environ.get("QUANT002_SN")
CLARITY_APIKEY = os.environ.get("CLARITY_APIKEY")
CLARITY_ORGID = os.environ.get("CLARITY_ORGID")


In [5]:
def get_quantaq_data(url_ending):
    """
    A function that takes a list of words that go immediately after
    after the "https://api.quant-aq.com" of the url and help to 
    get access to scrap data from quantaq

    Args: 
        url_list (list):    A list of strings to be added to the url 
    
    Returns:
        response_dict (dictionary): a dictionary with the requested data        
    """

    #Define the fist part of the url
    url_first_part = "https://api.quant-aq.com/"

    #Initialize a continieoulsy increasing string to be used to complete the url
    url_additional = ""

    #Complete the url with user input strings 
    for adds_on in url_ending:
        url_additional = url_additional+adds_on+"/"        
    complete_url = url_first_part+url_additional

    #Make the call of the response to be returned 
    response_json = requests.get(complete_url, auth=(QUANTAQ_APIKEY,""))

    #Convert to dictionary for easy manipulation in python
    response_dict = json.loads(response_json.content)

    return response_dict

In [46]:
def upload_to_inlfuxdb(data_frame_to_save, measurement_name, index_field, client_name):
    """
    A function that takes the name of the measurement(table) 
    and one field name to serve as the index and wites them into
    a defined database on influxdb

    Args:
        data_frame_to_save(df)  : The dataframe to be sent to the database
        measurement_name(string) : The measurement or table name to store the dataset
        index_field(string)     : A timefield from the measurement to be used as an index
                                This also serves as the timeseries reference column
        client_name(object)     : The influxdb object with a defined database 

    Return : 
            Prints that it completed the task
    """ 

    #Make the index field to be the index so that it acts as the timestamp
    data_frame_to_save_index = data_frame_to_save.set_index(index_field)
    
    #Write to the database and the let user know that it is written
    client_name.write(record=data_frame_to_save_index, 
                      data_frame_measurement_name=measurement_name)
    
    print("Done! Check your influxdb to confrim!")



Transform the quantaaq data obtained through the API into a datframe, and send it to the influxdb database for further querrying and vizualization

In [47]:
#Get the json data to be saved to influxdb from the quantaq
data_meta_quant002_dict = get_quantaq_data(['v1','devices',QUANT002_SN,'data'])

#Extract the data part of the transformed data 
# for easy saving to the influxdb database
data_quant002_dict = data_meta_quant002_dict['data']

#Transform the data to a dataframe for 
# easy compatibility with influxdb classes
data_quant002_df = pd.DataFrame(data=data_quant002_dict)

#Use the upload function to send this datframe to inlflux
upload_to_inlfuxdb(data_frame_to_save=data_quant002_df, 
                   measurement_name='quantaq-pilot', 
                   index_field='timestamp', 
                   client_name=client)

Done! Check your influxdb to confrim!


In [ ]:
#Get the json data to be saved to influxdb from the quantaq
data_meta_quant002_dict = get_quantaq_data(['v1','devices',QUANT002_SN,'data'])

#Extract the data part of the transformed data 
# for easy saving to the influxdb database
data_quant002_dict = data_meta_quant002_dict['data']

#Transform the data to a dataframe for 
# easy compatibility with influxdb classes
data_quant002_df = pd.DataFrame(data=data_quant002_dict)

#Set the UTC timestamp to be the index for direct querrying
data_quant002_df_index = data_quant002_df.set_index('timestamp')

#Define the table name where the data will be saved in influxsb
measurement_name_quantaq = "quantaq-pilot"

#Save the data to influxdb to allow querying and future manipulation
client.write(record=data_quant002_df_index, 
             data_frame_measurement_name = measurement_name_quantaq)

#### 2. Clarity
The general guidelines for accessing clarity data through the api can be found [here](https://api-guide.clarity.io/); These guideline use examples based on POSTMAN - a webtool for obtaining data using APIs. Clarity has a python library for retrieving data found [here](https://github.com/a2gov/clarityio). Here we use the python library.

First install the clarity library

In [8]:
%%skip
pip install clarityio

Make initial connections with the clarity library


In [ ]:
import clarityio # Offers methods to connect to the clarity api
from io import StringIO  # Helps to read file as csv and transform to df
api_connection = clarityio.ClarityAPIConnection(api_key=CLARITY_APIKEY, org=CLARITY_ORGID)

Getting the data from the APi before uplaoding to the database

In [53]:
#Attemopt to get data from a certain date
request_body_pilot = {
    'allDatasources': True,
    'outputFrequency': 'hour',
    'format':       'csv-wide',
    'startTime': '2026-01-04T00:00:00Z' 
    }

#get the data in form of csv, ocnvert it to a daframe and 
# for easy manipulation and saving to the database
response_wide2 = api_connection.get_recent_measurements(data=request_body_pilot)
df_clarity_pilot = pd.read_csv(StringIO(response_wide2), sep=",")

#Ensure that only meaningful columns exist to allow 
# easy data search and manipulation in influxdb
df_clarity_pilot_db = df_clarity_pilot.dropna(axis='columns', how='all')
df_clarity_pilot_db


,datasourceId,sourceId,sourceType,outputFrequency,startOfPeriod,endOfPeriod,locationLatitude,locationLongitude,no2Conc1HourMean.raw,no2Conc1HourMean.status,...,pm2_5ConcNum1HourMean.value,pm2_5ConcNum24HourRollingMean.raw,pm2_5ConcNum24HourRollingMean.status,pm2_5ConcNum24HourRollingMean.value,relHumidInternal1HourMean.raw,relHumidInternal1HourMean.status,relHumidInternal1HourMean.value,temperatureInternal1HourMean.raw,temperatureInternal1HourMean.status,temperatureInternal1HourMean.value
0,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-07T20:00:00Z,2026-01-07T21:00:00Z,40.315627,-76.895969,11.23,calibration-missing,...,7.60,53.79,sensor-ready,53.79,51.44,sensor-ready,51.44,14.98,sensor-ready,14.98
1,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-07T20:00:00Z,2026-01-07T21:00:00Z,40.264301,-76.851220,15.86,calibration-missing,...,5.87,42.04,sensor-ready,42.04,45.66,sensor-ready,45.66,15.48,sensor-ready,15.48
2,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-07T20:00:00Z,2026-01-07T21:00:00Z,40.315625,-76.895981,12.14,calibration-missing,...,7.28,49.47,sensor-ready,49.47,50.28,sensor-ready,50.28,15.08,sensor-ready,15.08
3,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-07T19:00:00Z,2026-01-07T20:00:00Z,40.315627,-76.895969,-1.10,calibration-missing,...,8.25,56.70,sensor-ready,56.70,50.05,sensor-ready,50.05,16.59,sensor-ready,16.59
4,DSCFB3568,A47HWW6V,CLARITY_NODE,hour,2026-01-07T19:00:00Z,2026-01-07T20:00:00Z,40.264301,-76.851220,5.92,calibration-missing,...,7.06,44.35,sensor-ready,44.35,43.99,sensor-ready,43.99,16.78,sensor-ready,16.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-04T22:00:00Z,2026-01-04T23:00:00Z,37.879027,-122.301929,9.52,calibration-missing,...,14.24,21.17,sensor-ready,21.17,59.52,sensor-ready,59.52,-0.08,sensor-ready,-0.08
187,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-04T21:00:00Z,2026-01-04T22:00:00Z,37.879027,-122.301929,10.09,calibration-missing,...,14.81,21.53,sensor-ready,21.53,58.22,sensor-ready,58.22,0.32,sensor-ready,0.32
188,DALCU7773,A93NM49L,CLARITY_NODE,hour,2026-01-04T21:00:00Z,2026-01-04T22:00:00Z,37.879027,-122.301929,7.13,calibration-missing,...,14.02,21.33,sensor-ready,21.33,57.88,sensor-ready,57.88,0.48,sensor-ready,0.48
189,DYITV0908,A44MFTF3,CLARITY_NODE,hour,2026-01-04T20:00:00Z,2026-01-04T21:00:00Z,37.879027,-122.301929,9.69,calibration-missing,...,14.60,22.09,sensor-ready,22.09,57.70,sensor-ready,57.70,0.57,sensor-ready,0.57


In [54]:
#Use the upload function to send this datframe to inlflux
upload_to_inlfuxdb(data_frame_to_save=df_clarity_pilot_db, 
                   measurement_name='clarity-pilot', 
                   index_field='endOfPeriod', 
                   client_name=client)

Done! Check your influxdb to confrim!


In [ ]:
#Test clarity

test_url_clarity = ['v2', 'devices', 'nodes?org=communUEVL']
test2_url_clarity = ['v2', 'recent-datasource-measurements-query']
test3_url_clarity = ['v2', 'report-requests']

get_clarity_data(test_url_clarity)


[{'nodeId': 'A43H6HKC',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [],
  'lifeStage': {'stage': 'purchased', 'when': '2025-10-30T18:16:32.946Z'}},
 {'nodeId': 'A636FW6Q',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [],
  'lifeStage': {'stage': 'purchased', 'when': '2025-10-30T18:16:33.168Z'}},
 {'nodeId': 'A63CMY3J',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [],
  'lifeStage': {'stage': 'purchased', 'when': '2025-10-30T18:16:33.266Z'}},
 {'nodeId': 'A44MFTF3',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [{'accessoryId': 'MF99PJYR',
    'sku': 'CLA-MGAS-A01'}],
  'lifeStage': {'stage': 'working', 'when': '2025-11-07T18:15:33.271Z'},
  'location': {'type': 'Point',
   'coordinates': [-122.30192899608234, 37.87902695401437]}},
 {'nodeId': 'A47HWW6V',
  'model': 'Node-S Global',
  'sku': 'CLA02-00001',
  'pairedAccessoryModules': [{'accessoryId'

In [108]:
headers_clarity = {
        'X-API-key' : CLARITY_APIKEY,
        'Accept' : "application/json"
    }
requests.get('https://clarity-data-api.clarity.io/v2/report-requests', headers=headers_clarity).content

b'{"message": "Resource not found. Please, check the URL for typos."}'